# Experimento 6: BioLinkBERT-base para Relation Extraction

**Objetivo:** Comparar BioLinkBERT-base con PubMedBERT usando exactamente la misma configuracion que el Experimento 1.

**Hipotesis:** BioLinkBERT fue entrenado no solo con texto biomedico sino tambien modelando los **enlaces** entre documentos de PubMed (como si supiera que articulos citan a otros). Esto le da una comprension mas rica de las conexiones entre conceptos, lo que deberia ser especialmente util para relation extraction.

**Diferencia con Exp1:** Solo cambia el backbone. Todo lo demas (datos, hiperparametros, arquitectura) es identico para que la comparacion sea justa.

**Resultados anteriores:**
- Baseline (bert-multilingual, 10ep): Macro F1 = 0.6944
- Exp1: PubMedBERT (10ep): Macro F1 = 0.7754
- Exp2: PubMedBERT + typed markers (15ep): Macro F1 = 0.8078
- Exp3: PubMedBERT + typed markers + neg_ratio 1:1 (15ep): Macro F1 = 0.8430

**Modelo a probar:** `michiyasunaga/BioLinkBERT-base`

**Referencia bibliografica:** Sanger & Leser, BMC Bioinformatics 2025 — BioLinkBERT-large es el mejor modelo en todos los datasets de RE biomedica evaluados.

## 1. Setup e Instalacion

In [ ]:
# Ejecucion en servidor remoto (zape), entorno conda "tfg".
# Descomenta la primera vez que uses este entorno (instala dependencias):
# !pip install git+https://github.com/thunlp/OpenNRE.git
# !pip install torch transformers nltk pandas scikit-learn matplotlib seaborn tqdm

# NOTA: al reiniciar el kernel es normal ver un banner de /etc/profile diciendo
# "All Hugging Face models are cached in /opt/huggingface" -- es ruido informativo
# del login shell del servidor, no significa que el override de abajo no vaya a
# aplicarse. Basta con ejecutar esta celda ANTES de cualquier import de
# opennre/transformers/huggingface_hub para que quede sobrescrito el resto de la sesion.

# La cache compartida /opt/huggingface no es escribible para este usuario
# (no pertenece al grupo "users" que la posee). El servidor ya trae HF_HOME
# / HF_HUB_CACHE definidas globalmente apuntando ahi, asi que hace falta
# SOBRESCRIBIRLAS (no basta con setdefault) para que apunten a una carpeta
# propia del home donde si podemos descargar modelos nuevos.
import os
HF_CACHE_DIR = os.path.expanduser("~/hf_cache")
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
print(f"HF_HOME: {os.environ['HF_HOME']}")
print(f"HF_HUB_CACHE: {os.environ['HF_HUB_CACHE']}")

# Verificacion: confirma que huggingface_hub (no solo os.environ) resuelve la
# cache al directorio propio, y que es escribible. Si esto falla, algo importo
# huggingface_hub/transformers ANTES de esta celda en la sesion actual.
import huggingface_hub.constants as hfc
assert hfc.HF_HUB_CACHE == os.environ["HF_HUB_CACHE"], (
    f"huggingface_hub resolvio {hfc.HF_HUB_CACHE!r}, se esperaba {os.environ['HF_HUB_CACHE']!r}. "
    "Reinicia el kernel y ejecuta esta celda primero, antes de cualquier import de HF/opennre."
)
test_file = os.path.join(os.environ["HF_HUB_CACHE"], ".write_test")
with open(test_file, "w") as f:
    f.write("ok")
os.remove(test_file)
print("Cache HF verificada y escribible.")

# Aplica los parches de compatibilidad de OpenNRE
# (UTF-8 en lectura de datos, AdamW de torch en vez de transformers, num_workers=0)
!python ../baseline/patch_opennre.py

In [ ]:
import json
import importlib
import time
import logging
import os
from collections import Counter
from pathlib import Path

import nltk
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import opennre

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuracion del Experimento

La unica diferencia con Exp1 es `MODEL_NAME`. Todo lo demas es identico.

In [ ]:
# ============================================================
# CONFIGURACION - EXPERIMENTO 6: BioLinkBERT-base
# ============================================================

# *** UNICO CAMBIO RESPECTO A EXP1 ***
MODEL_NAME = "michiyasunaga/BioLinkBERT-base"
EXPERIMENT_NAME = "biolinkbert_base"

# Por que BioLinkBERT?
# - Entrenado con PubMed + enlaces entre documentos (document linking)
# - Entiende conexiones entre conceptos biomedicos mejor que PubMedBERT
# - Sanger & Leser 2025 (BMC Bioinformatics): mejor modelo en 5/5 datasets de RE biomedica
# - La version base tiene ~110M params (igual que PubMedBERT), sin problemas de VRAM en T4

# HIPERPARAMETROS (identicos a Exp1, 15 epochs para comparacion justa)
MAX_LENGTH = 256
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
EPOCHS = 15
WARMUP_STEPS = 300
SEED = 42
NEG_RATIO = 3  # igual que Exp1, datos originales sin modificar

# Paths LOCALES (ejecucion en local, no Kaggle)
DATA_DIR = Path("../data/english")
TRAIN_DATA = DATA_DIR / "eng_train.txt"
DEV_DATA = DATA_DIR / "eng_dev.txt"
REL2ID_PATH = DATA_DIR / "rel2id.json"

# Outputs: carpeta propia de este experimento
OUTPUT_DIR = Path("../outputs/1B-biolinkbert")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = OUTPUT_DIR / f"eng_{EXPERIMENT_NAME}.pth.tar"
PRED_PATH = OUTPUT_DIR / f"eng_pred_{EXPERIMENT_NAME}.tsv"

RELATION_TYPES = [
    "ABBREVIATION", "ALTERNATIVE_NAME", "SUBCLASS_OF", "PART_OF",
    "TREATED_USING", "ORIGINS_FROM", "TO_DETECT_OR_STUDY", "AFFECTS",
    "HAS_CAUSE", "APPLIED_TO", "USED_IN", "ASSOCIATED_WITH",
    "PHYSIOLOGY_OF", "FINDING_OF",
    "no_relation",
]

with open(REL2ID_PATH) as f:
    rel2id = json.load(f)

print(f"Experimento: {EXPERIMENT_NAME}")
print(f"Modelo: {MODEL_NAME}")
print(f"Datos: {DATA_DIR}")
print(f"Checkpoint: {CKPT_PATH}")
print(f"Clases: {len(RELATION_TYPES)} (14 relaciones + no_relation)")

## 3. Verificar Datos

In [ ]:
assert TRAIN_DATA.exists(), f"No se encuentra {TRAIN_DATA}"
assert DEV_DATA.exists(), f"No se encuentra {DEV_DATA}"
assert REL2ID_PATH.exists(), f"No se encuentra {REL2ID_PATH}"

with open(TRAIN_DATA) as f:
    n_train = sum(1 for line in f if line.strip())
with open(DEV_DATA) as f:
    n_dev = sum(1 for line in f if line.strip())

print(f"Train: {n_train} instancias")
print(f"Dev:   {n_dev} instancias")
print(f"Clases: {len(rel2id)}")

# Distribucion de relaciones
train_instances = []
with open(TRAIN_DATA, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            train_instances.append(json.loads(line))

rel_counts = Counter(inst["relation"] for inst in train_instances)
print(f"\nDistribucion en train:")
for rel, count in rel_counts.most_common():
    print(f"  {rel:<25} {count:>6} ({100*count/len(train_instances):5.1f}%)")

## 4. Parchear OpenNRE y Entrenar

Mismos parches que en Exp1: AdamW de torch en vez de transformers, y `dataset.eval()` extendido para calcular macro_f1 y seleccionar el mejor checkpoint por esa métrica (no por micro_f1).

In [ ]:
# Parchear OpenNRE
import opennre.framework.sentence_re as sre

sre_path = Path(sre.__file__)
sre_text = sre_path.read_text(encoding="utf-8")
sre_text = sre_text.replace(
    "from transformers import AdamW",
    "from torch.optim import AdamW",
)
sre_text = sre_text.replace(
    "self.optimizer = AdamW(grouped_params, correct_bias=False)",
    "self.optimizer = AdamW(grouped_params)",
)
sre_path.write_text(sre_text, encoding="utf-8")
importlib.reload(sre)
opennre.framework.SentenceRE = sre.SentenceRE
print("OpenNRE parcheado.")

In [ ]:
# ============================================================
# Parchear OpenNRE: dataset.eval() tambien calcula macro_f1
# (por defecto solo calcula acc/micro_p/micro_r/micro_f1; el bucle de
# entreno de abajo selecciona el mejor checkpoint segun una clave que
# exista en ese dict). Con esto seleccionamos por Macro F1 (la metrica
# que reportamos en todo el TFG) en vez de por micro_f1 -- igual que Exp1.
# ============================================================
import opennre.framework.data_loader as dl
from collections import Counter as _Counter

_orig_dataset_eval = dl.SentenceREDataset.eval

def _eval_with_macro(self, pred_result, use_name=False):
    result = _orig_dataset_eval(self, pred_result, use_name=use_name)

    neg_names = {"NA", "na", "no_relation", "Other", "Others"}
    id2rel_local = {v: k for k, v in self.rel2id.items()}

    gold = [
        (self.data[i]["relation"] if use_name else self.rel2id[self.data[i]["relation"]])
        for i in range(len(self.data))
    ]

    tp, fp, fn, support = _Counter(), _Counter(), _Counter(), _Counter()
    for g, p in zip(gold, pred_result):
        support[g] += 1
        if g == p:
            tp[g] += 1
        else:
            fp[p] += 1
            fn[g] += 1

    f1s = []
    for key, n_support in support.items():
        name = key if use_name else id2rel_local[key]
        if name in neg_names or n_support == 0:
            continue
        p_ = tp[key] / (tp[key] + fp[key]) if (tp[key] + fp[key]) else 0
        r_ = tp[key] / (tp[key] + fn[key]) if (tp[key] + fn[key]) else 0
        f1 = 2 * p_ * r_ / (p_ + r_) if (p_ + r_) else 0
        f1s.append(f1)

    result["macro_f1"] = sum(f1s) / len(f1s) if f1s else 0.0
    return result

dl.SentenceREDataset.eval = _eval_with_macro
print("dataset.eval() parcheado: ahora devuelve tambien 'macro_f1'.")

In [ ]:
print("=" * 60)
print(f"ENTRENAMIENTO: {EXPERIMENT_NAME}")
print(f"Modelo: {MODEL_NAME}")
print(f"Datos: originales (neg_ratio=3:1, sin typed markers)")
print(f"Epochs: {EPOCHS}")
print("=" * 60)

encoder = opennre.encoder.BERTEntityEncoder(
    max_length=MAX_LENGTH,
    pretrain_path=MODEL_NAME,
)

model = opennre.model.SoftmaxNN(
    sentence_encoder=encoder,
    num_class=len(rel2id),
    rel2id=rel2id,
)

framework = opennre.framework.SentenceRE(
    model=model,
    train_path=str(TRAIN_DATA),
    val_path=str(DEV_DATA),
    test_path=str(DEV_DATA),
    ckpt=str(CKPT_PATH),
    batch_size=BATCH_SIZE,
    max_epoch=EPOCHS,
    lr=LEARNING_RATE,
    opt="adamw",
    warmup_step=WARMUP_STEPS,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nParametros del modelo: {n_params:,}")
print(f"Listo para entrenar.")

In [ ]:
# ============================================================
# ENTRENAR con historial por epoch (loss/acc train + P/R/F1 val)
# Reimplementa el bucle interno de SentenceRE.train_model (que no expone
# metricas por epoch) para poder guardar una curva de entreno y validar
# en cada epoch. Guarda el mejor checkpoint segun METRIC (macro_f1).
# ============================================================
from opennre.framework.utils import AverageMeter
from tqdm import tqdm

METRIC = "macro_f1"

def train_with_history(fw, max_epoch, metric="macro_f1"):
    history = []
    best_metric = 0
    for epoch in range(max_epoch):
        fw.train()
        avg_loss = AverageMeter()
        avg_acc = AverageMeter()
        t = tqdm(fw.train_loader, desc=f"Epoch {epoch}")
        for data in t:
            if torch.cuda.is_available():
                for i in range(len(data)):
                    try:
                        data[i] = data[i].cuda()
                    except Exception:
                        pass
            label = data[0]
            args = data[1:]
            logits = fw.parallel_model(*args)
            loss = fw.criterion(logits, label)
            _, pred = logits.max(-1)
            acc = float((pred == label).long().sum()) / label.size(0)
            avg_loss.update(loss.item(), 1)
            avg_acc.update(acc, 1)
            t.set_postfix(loss=avg_loss.avg, acc=avg_acc.avg)
            loss.backward()
            fw.optimizer.step()
            if fw.scheduler is not None:
                fw.scheduler.step()
            fw.optimizer.zero_grad()

        # Validacion al final de cada epoch (dataset.eval ya devuelve macro_f1, celda parcheada arriba)
        val_result = fw.eval_model(fw.val_loader)

        record = {
            "epoch": epoch,
            "train_loss": avg_loss.avg,
            "train_acc": avg_acc.avg,
            "val_acc": val_result["acc"],
            "val_micro_p": val_result["micro_p"],
            "val_micro_r": val_result["micro_r"],
            "val_micro_f1": val_result["micro_f1"],
            "val_macro_f1": val_result["macro_f1"],
        }
        history.append(record)
        print(f"Epoch {epoch}: train_loss={record['train_loss']:.4f} train_acc={record['train_acc']:.4f} "
              f"val_micro_f1={record['val_micro_f1']:.4f} val_macro_f1={record['val_macro_f1']:.4f}")

        if val_result[metric] > best_metric:
            print(f"  -> nuevo mejor {metric}={val_result[metric]:.4f}, guardando checkpoint")
            folder_path = "/".join(fw.ckpt.split("/")[:-1])
            if folder_path and not os.path.exists(folder_path):
                os.makedirs(folder_path, exist_ok=True)
            torch.save({"state_dict": fw.model.state_dict()}, fw.ckpt)
            best_metric = val_result[metric]

    print(f"\nMejor {metric} en val: {best_metric:.4f}")
    return history

# ENTRENAR - Esta celda tarda ~60-90 min con GPU (15 epochs)
start_time = time.time()

history = train_with_history(framework, EPOCHS, metric=METRIC)

elapsed = time.time() - start_time
print(f"\nEntrenamiento completado en {elapsed/60:.1f} minutos")
print(f"Mejor checkpoint guardado en: {CKPT_PATH}")

HISTORY_PATH = OUTPUT_DIR / f"history_{EXPERIMENT_NAME}.json"
with open(HISTORY_PATH, "w") as f:
    json.dump(history, f, indent=2)
print(f"Historial guardado en: {HISTORY_PATH}")

In [ ]:
# ============================================================
# GRAFICA DE LA CURVA DE ENTRENO (loss + F1 por epoch)
# ============================================================
epochs_x = [h["epoch"] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Curva de entrenamiento - {EXPERIMENT_NAME}", fontsize=13, fontweight="bold")

ax = axes[0]
ax.plot(epochs_x, [h["train_loss"] for h in history], marker="o", color="#e74c3c", label="train_loss")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_title("Loss de entrenamiento")
ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(epochs_x, [h["train_acc"] for h in history], marker="o", label="train_acc")
ax2.plot(epochs_x, [h["val_micro_f1"] for h in history], marker="s", label="val_micro_f1")
ax2.plot(epochs_x, [h["val_macro_f1"] for h in history], marker="^", label="val_macro_f1")
best_epoch = max(history, key=lambda h: h["val_macro_f1"])["epoch"]
ax2.axvline(best_epoch, color="gray", linestyle="--", alpha=0.6, label=f"mejor epoch (macro_f1) = {best_epoch}")
ax2.set_xlabel("epoch"); ax2.set_ylabel("score"); ax2.set_title("Accuracy / F1 por epoch")
ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / f"curva_entreno_{EXPERIMENT_NAME}.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Grafica guardada en: {OUTPUT_DIR / f'curva_entreno_{EXPERIMENT_NAME}.png'}")

In [ ]:
# ============================================================
# COMPARACION: que epoch habria elegido micro_f1 vs. macro_f1
# (justificacion para la memoria del TFG de por que seleccionar por macro_f1)
# ============================================================
best_by_micro = max(history, key=lambda h: h["val_micro_f1"])
best_by_macro = max(history, key=lambda h: h["val_macro_f1"])

print(f"{'Criterio de seleccion':<30}{'Epoch':>8}{'val_micro_f1':>16}{'val_macro_f1':>16}")
print("-" * 70)
print(f"{'Si se seleccionara por micro_f1':<30}{best_by_micro['epoch']:>8}{best_by_micro['val_micro_f1']:>16.4f}{best_by_micro['val_macro_f1']:>16.4f}")
print(f"{'Seleccionado por macro_f1 (usado)':<30}{best_by_macro['epoch']:>8}{best_by_macro['val_micro_f1']:>16.4f}{best_by_macro['val_macro_f1']:>16.4f}")
print("-" * 70)

if best_by_micro["epoch"] == best_by_macro["epoch"]:
    print("\nEn este run coinciden en la misma epoch: no hay divergencia que mostrar esta vez.")
else:
    diff = best_by_macro["val_macro_f1"] - best_by_micro["val_macro_f1"]
    print(f"\nDivergencia real: seleccionar por micro_f1 habria dado macro_f1={best_by_micro['val_macro_f1']:.4f} "
          f"(epoch {best_by_micro['epoch']}), frente a macro_f1={best_by_macro['val_macro_f1']:.4f} "
          f"(epoch {best_by_macro['epoch']}) seleccionando por macro_f1 -> diferencia de {diff:+.4f}.")

json.dump({
    "best_by_micro_f1": best_by_micro,
    "best_by_macro_f1": best_by_macro,
}, open(OUTPUT_DIR / f"comparacion_micro_vs_macro_{EXPERIMENT_NAME}.json", "w"), indent=2)
print(f"\nguardado: comparacion_micro_vs_macro_{EXPERIMENT_NAME}.json")

## 5. Prediccion en Dev

In [ ]:
# Cargar mejor checkpoint
encoder_pred = opennre.encoder.BERTEntityEncoder(
    max_length=MAX_LENGTH,
    pretrain_path=MODEL_NAME,
)

model_pred = opennre.model.SoftmaxNN(
    sentence_encoder=encoder_pred,
    num_class=len(rel2id),
    rel2id=rel2id,
)

ckpt = torch.load(str(CKPT_PATH), map_location="cpu")
model_pred.load_state_dict(ckpt["state_dict"])

if torch.cuda.is_available():
    model_pred = model_pred.cuda()
model_pred.eval()

print(f"Modelo cargado desde: {CKPT_PATH}")

In [ ]:
# Predecir en dev
dev_instances = []
with open(DEV_DATA, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            dev_instances.append(json.loads(line))

print(f"Prediciendo {len(dev_instances)} instancias...")

rows = []
for inst in dev_instances:
    pred_rel, score = model_pred.infer({
        "text": inst["text"],
        "h": {"pos": inst["h"]["pos"]},
        "t": {"pos": inst["t"]["pos"]},
    })
    rows.append({
        "document_id": inst["doc_id"],
        "relation": pred_rel,
        "score": score,
        "gold": inst["relation"],
        "head_text": inst["h"]["name"],
        "head_span": inst["head_span"],
        "head_type": inst["head_type"],
        "tail_text": inst["t"]["name"],
        "tail_span": inst["tail_span"],
        "tail_type": inst["tail_type"],
    })

# Guardar predicciones (formato CodaBench)
pred_df = pd.DataFrame(rows)
pred_export = pred_df[["document_id", "relation", "head_text", "head_span",
                        "head_type", "tail_text", "tail_span", "tail_type"]].copy()
pred_export = pred_export[pred_export["relation"] != "no_relation"]
pred_export.to_csv(PRED_PATH, sep="\t", index=False)

n_filtered = len(pred_df) - len(pred_export)
print(f"Filtradas {n_filtered} predicciones no_relation")
print(f"Predicciones guardadas en: {PRED_PATH}")

## 6. Evaluacion y Comparacion

In [ ]:
# Calcular metricas por relacion
all_relations = sorted(rel2id.keys())

gold_labels = [inst["relation"] for inst in dev_instances]
pred_labels = [row["relation"] for row in rows]

correct = sum(1 for g, p in zip(gold_labels, pred_labels) if g == p)
total = len(gold_labels)
accuracy = correct / total

gold_counts = Counter(gold_labels)
pred_counts = Counter(pred_labels)
tp_counts = Counter()
for g, p in zip(gold_labels, pred_labels):
    if g == p:
        tp_counts[g] += 1

results = {}
print(f"{'Relacion':<25} {'P':>8} {'R':>8} {'F1':>8} {'Soporte':>8}")
print("-" * 60)

f1_scores = []
for rel in all_relations:
    tp = tp_counts.get(rel, 0)
    pred_total = pred_counts.get(rel, 0)
    gold_total = gold_counts.get(rel, 0)
    p = tp / pred_total if pred_total else 0
    r = tp / gold_total if gold_total else 0
    f1 = 2 * p * r / (p + r) if (p + r) else 0
    results[rel] = {"precision": p, "recall": r, "f1": f1, "support": gold_total}
    if gold_total > 0:
        f1_scores.append(f1)
        print(f"{rel:<25} {p:>8.4f} {r:>8.4f} {f1:>8.4f} {gold_total:>8}")

macro_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0

print("-" * 60)
print(f"\nAccuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

# Tabla comparativa completa
print(f"\n{'='*65}")
print(f"COMPARACION DE TODOS LOS EXPERIMENTOS")
print(f"{'='*65}")
print(f"{'Experimento':<45} {'Macro F1':>10} {'Diff':>10}")
print(f"{'-'*65}")
print(f"{'Baseline (bert-multi, 10ep)':<45} {'0.6944':>10} {'---':>10}")
print(f"{'Exp1: PubMedBERT (10ep)':<45} {'0.7754':>10} {'+0.0810':>10}")
print(f"{'Exp2: PubMedBERT + typed markers (15ep)':<45} {'0.8078':>10} {'+0.1134':>10}")
print(f"{'Exp3: PubMedBERT + typed + neg 1:1 (15ep)':<45} {'0.8430':>10} {'+0.1486':>10}")
print(f"{'Exp6: BioLinkBERT-base (' + str(EPOCHS) + 'ep)':<45} {macro_f1:>10.4f} {macro_f1 - 0.6944:>+10.4f}")
print(f"{'='*65}")
print(f"\nBioLinkBERT-base vs PubMedBERT (Exp1): {macro_f1 - 0.7754:+.4f}")

## 7. Analisis de Errores

In [ ]:
# Errores mas comunes
confusion_pairs = Counter()
for g, p in zip(gold_labels, pred_labels):
    if g != p:
        confusion_pairs[(g, p)] += 1

total_errors = sum(confusion_pairs.values())
print(f"Total errores: {total_errors} / {total} ({100*total_errors/total:.1f}%)")

# Comparar con los mismos errores en Exp1 (PubMedBERT)
exp1_errors = {
    ("ALTERNATIVE_NAME", "no_relation"): 53,
    ("HAS_CAUSE", "no_relation"): 53,
    ("PART_OF", "SUBCLASS_OF"): 51,
    ("AFFECTS", "no_relation"): 36,
    ("SUBCLASS_OF", "HAS_CAUSE"): 26,
}

print(f"\nComparacion con Exp1 (PubMedBERT) en las confusiones clave:")
print(f"{'Confusion':<50} {'Exp1':>6} {'Exp6':>6} {'Cambio':>8}")
print("-" * 75)
for (g, p), prev_count in sorted(exp1_errors.items(), key=lambda x: -x[1]):
    curr_count = confusion_pairs.get((g, p), 0)
    diff = curr_count - prev_count
    symbol = "✓ mejora" if diff < 0 else "✗ peor" if diff > 0 else "= igual"
    print(f"{g} -> {p:<30} {prev_count:>6} {curr_count:>6} {diff:>+8} ({symbol})")

print(f"\nTop 10 confusiones actuales (BioLinkBERT):")
print(f"{'Real':<25} {'Predicho':<25} {'Count':>6}")
print("-" * 60)
for (g, p), count in confusion_pairs.most_common(10):
    print(f"{g:<25} {p:<25} {count:>6}")

In [ ]:
# Comparacion visual: BioLinkBERT vs PubMedBERT por relacion
# F1 de Exp1 (PubMedBERT 10ep)
pubmedbert_f1 = {
    "ABBREVIATION": 0.9200,
    "AFFECTS": 0.8796,
    "ALTERNATIVE_NAME": 0.3465,
    "APPLIED_TO": 0.5743,
    "ASSOCIATED_WITH": 0.8107,
    "FINDING_OF": 0.6821,
    "HAS_CAUSE": 0.8179,
    "ORIGINS_FROM": 0.8571,
    "PART_OF": 0.7862,
    "PHYSIOLOGY_OF": 0.8571,
    "SUBCLASS_OF": 0.8728,
    "TO_DETECT_OR_STUDY": 0.8072,
    "TREATED_USING": 0.9024,
    "USED_IN": 0.7421,
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Experimento 6: BioLinkBERT-base vs PubMedBERT", fontsize=14, fontweight="bold")

rels_plot = [r for r in all_relations if results.get(r, {}).get("support", 0) > 0]
f1_pubmed = [pubmedbert_f1.get(r, 0) for r in rels_plot]
f1_biolink = [results[r]["f1"] for r in rels_plot]

# Barras comparativas
x = np.arange(len(rels_plot))
width = 0.35
axes[0].bar(x - width/2, f1_pubmed, width,
            label=f'PubMedBERT (F1=0.7754)', color='#3498db', alpha=0.8)
axes[0].bar(x + width/2, f1_biolink, width,
            label=f'BioLinkBERT-base (F1={macro_f1:.4f})', color='#e74c3c', alpha=0.8)
axes[0].set_xlabel('Relacion')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('F1 por relacion')
axes[0].set_xticks(x)
axes[0].set_xticklabels(rels_plot, rotation=45, ha='right', fontsize=8)
axes[0].legend()
axes[0].set_ylim(0, 1)

# Diferencia por relacion
diffs = [f1_biolink[i] - f1_pubmed[i] for i in range(len(rels_plot))]
colors = ['#2ecc71' if d > 0 else '#e74c3c' for d in diffs]
axes[1].barh(rels_plot, diffs, color=colors)
axes[1].set_xlabel('Cambio en F1 (BioLinkBERT - PubMedBERT)')
axes[1].set_title('Impacto de cambiar backbone')
axes[1].axvline(x=0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / f"comparacion_{EXPERIMENT_NAME}.png"), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matriz de confusion
from sklearn.metrics import confusion_matrix

gold_idx = [rel2id[g] for g in gold_labels]
pred_idx = [rel2id[p] for p in pred_labels]

top_rels = [r for r, _ in Counter(gold_labels).most_common(8)]
top_indices = [rel2id[r] for r in top_rels]

cm = confusion_matrix(gold_idx, pred_idx, labels=top_indices)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=top_rels, yticklabels=top_rels)
plt.title(f'Matriz de Confusion - {EXPERIMENT_NAME} (Top 8 relaciones)')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / f"confusion_{EXPERIMENT_NAME}.png"), dpi=150, bbox_inches='tight')
plt.show()

## 8. Guardar Resultados

In [ ]:
experiment_results = {
    "experiment": EXPERIMENT_NAME,
    "model": MODEL_NAME,
    "technique": "solo cambio de backbone (datos originales, sin typed markers)",
    "hyperparameters": {
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "warmup_steps": WARMUP_STEPS,
        "neg_ratio": NEG_RATIO,
        "seed": SEED,
    },
    "results": {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "per_relation": results,
    },
    "comparison": {
        "baseline_macro_f1": 0.6944,
        "pubmedbert_10ep_macro_f1": 0.7754,
        "pubmedbert_typed_markers_macro_f1": 0.8078,
        "pubmedbert_typed_neg_ratio_macro_f1": 0.8430,
        "biolinkbert_base_macro_f1": macro_f1,
        "improvement_vs_baseline": macro_f1 - 0.6944,
        "improvement_vs_pubmedbert": macro_f1 - 0.7754,
    },
    "error_analysis": {
        "total_errors": total_errors,
        "error_rate": total_errors / total,
        "top_confusions": [
            {"gold": g, "pred": p, "count": c}
            for (g, p), c in confusion_pairs.most_common(10)
        ],
    },
    "n_train": n_train,
    "n_dev": n_dev,
}

results_path = OUTPUT_DIR / f"results_{EXPERIMENT_NAME}.json"
with open(results_path, "w") as f:
    json.dump(experiment_results, f, indent=2)

print(f"Resultados guardados en: {results_path}")
print(f"\nResumen final:")
print(f"  BioLinkBERT-base Macro F1: {macro_f1:.4f}")
print(f"  vs PubMedBERT: {macro_f1 - 0.7754:+.4f}")
print(f"  vs Baseline:   {macro_f1 - 0.6944:+.4f}")

## 9. Proximos Pasos

Dependiendo del resultado obtenido, hay dos direcciones posibles:

**Si BioLinkBERT-base supera a PubMedBERT:**
- Exp7: BioLinkBERT-base + typed markers + neg_ratio 1:1
  (aplicar las mismas mejoras de Exp2+Exp3 pero sobre este backbone)
- Exp8 (opcional): BioLinkBERT-**large** si la VRAM de Kaggle lo permite (333M params, batch_size=8)

**Si BioLinkBERT-base es similar o inferior a PubMedBERT:**
- Es un resultado igualmente valido para la memoria: demuestra que el dominio especifico
  (PubMedBERT) pesa mas que el preentrenamiento con links (BioLinkBERT)
- Continuar con Exp4 (focal loss) y Exp5 (class weights) sobre PubMedBERT

**Nota sobre BioLinkBERT-large:**
```python
# Para probar la version large (333M params):
MODEL_NAME = "michiyasunaga/BioLinkBERT-large"
EXPERIMENT_NAME = "biolinkbert_large"
BATCH_SIZE = 8  # reducir para caber en 15.6GB VRAM
```

---
## 10. Validación en modo blind (protocolo oficial CodaBench)

**Motivo:** `DEV_DATA` (dev curado, usado en las secciones 5-7) solo contiene los pares de entidades con relación anotada (0 casos de `no_relation`). El protocolo real de BioNNE-R —y como puntúa CodaBench en el test— enumera **todos los pares candidatos** de cada documento y deja que el modelo decida caso por caso. Esta sección reproduce ese protocolo sobre `eng_dev_blind.txt` (282.364 candidatos) reutilizando el checkpoint ya entrenado (`model_pred`, `encoder_pred`, `rel2id` de la sección 5).

**Referencia:** baseline oficial (`bert-base-multilingual-cased`) = 0.6944 curado / 0.3126-0.3435 ciego.

In [ ]:
# ============================================================
# 1) CARGAR DATOS BLIND
# ============================================================
BLIND_DEV_PATH = DATA_DIR / "eng_dev_blind.txt"
BLIND_GOLD_TSV = DATA_DIR / "eng-dev-rel.tsv"
BLIND_PRED_PATH = OUTPUT_DIR / f"eng_pred_blind_{EXPERIMENT_NAME}.tsv"
BLIND_RESULTS_PATH = OUTPUT_DIR / f"results_blind_{EXPERIMENT_NAME}.json"

for p in [BLIND_DEV_PATH, BLIND_GOLD_TSV]:
    print(f"{'OK ' if p.exists() else 'FALTA '} {p}")

blind_raw = [json.loads(l) for l in open(BLIND_DEV_PATH, encoding="utf-8") if l.strip()]
gold_df_blind = pd.read_csv(BLIND_GOLD_TSV, sep="\t")
id2rel = {v: k for k, v in rel2id.items()}

print(f"candidatos blind: {len(blind_raw)} | docs: {len(set(i['doc_id'] for i in blind_raw))} | gold real: {len(gold_df_blind)}")

In [ ]:
# ============================================================
# 2) INFERENCIA POR LOTES sobre los 282k candidatos
# model_pred.infer() de OpenNRE procesa 1 a 1; aqui batcheamos para
# que sea viable en tiempo (igual que en el resto de experimentos).
# ============================================================
N_GPUS = torch.cuda.device_count()
BLIND_BATCH_SIZE = 32 * max(1, N_GPUS)
device = "cuda" if torch.cuda.is_available() else "cpu"
infer_model = torch.nn.DataParallel(model_pred) if N_GPUS > 1 else model_pred
print(f"GPUs: {N_GPUS} | batch_size: {BLIND_BATCH_SIZE}")

def batched_predict(infer_model, encoder, instances, batch_size, device):
    preds, scores = [], []
    with torch.no_grad():
        for start in range(0, len(instances), batch_size):
            batch = instances[start:start + batch_size]
            tok = [encoder.tokenize({"text": i["text"], "h": {"pos": i["h"]["pos"]}, "t": {"pos": i["t"]["pos"]}}) for i in batch]
            fields = [torch.cat([t[k] for t in tok], dim=0).to(device) for k in range(len(tok[0]))]
            logits = infer_model(*fields)   # OJO: nunca infer_model.forward() con DataParallel
            probs = torch.softmax(logits, dim=-1)
            sc, pr = probs.max(-1)
            preds.extend(pr.tolist()); scores.extend(sc.tolist())
            done = min(start + batch_size, len(instances))
            if done % (batch_size * 50) == 0 or done == len(instances):
                print(f"  {done}/{len(instances)}", flush=True)
    return preds, scores

t0 = time.time()
blind_preds_id, blind_scores = batched_predict(infer_model, encoder_pred, blind_raw, BLIND_BATCH_SIZE, device)
blind_preds = [id2rel[p] for p in blind_preds_id]
elapsed_blind = time.time() - t0
print(f"inferencia: {elapsed_blind/60:.1f} min | {len(blind_raw)/elapsed_blind:.1f} inst/s")

In [ ]:
# ============================================================
# 3) CONSTRUIR PREDICCIONES (formato CodaBench)
# ============================================================
rows_blind = []
for inst, rel in zip(blind_raw, blind_preds):
    rows_blind.append({
        "document_id": inst["doc_id"], "relation": rel,
        "head_text": inst["h"]["name"], "head_span": inst["head_span"], "head_type": inst["head_type"],
        "tail_text": inst["t"]["name"], "tail_span": inst["tail_span"], "tail_type": inst["tail_type"],
    })
pred_df_blind = pd.DataFrame(rows_blind)
total_blind = len(pred_df_blind)
pred_df_blind = pred_df_blind[pred_df_blind["relation"] != "no_relation"]
print(f"filtradas {total_blind - len(pred_df_blind)} no_relation | quedan {len(pred_df_blind)}")
pred_df_blind.to_csv(BLIND_PRED_PATH, sep="\t", index=False)

In [ ]:
# ============================================================
# 4) EVALUAR CONTRA EL GOLD REAL Y COMPARAR CON EL DEV CURADO
# ============================================================
def _key(doc, hs, ts): return f"{doc}|{hs}|{ts}"

gold_rel = {_key(r.document_id, r.head_span, r.tail_span): r.relation for r in gold_df_blind.itertuples()}
pred_rel = {_key(r.document_id, r.head_span, r.tail_span): r.relation for r in pred_df_blind.itertuples()}

tp_b, fp_b, fn_b = Counter(), Counter(), Counter()
for k, pr in pred_rel.items():
    gr = gold_rel.get(k)
    if gr is None: fp_b[pr] += 1
    elif pr == gr: tp_b[pr] += 1
    else: fp_b[pr] += 1; fn_b[gr] += 1
for k, gr in gold_rel.items():
    if k not in pred_rel: fn_b[gr] += 1

rels_blind = sorted(set(gold_df_blind["relation"]))
gc_blind = Counter(gold_df_blind["relation"])
print(f"{'Relacion':<22}{'P':>8}{'R':>8}{'F1':>8}{'sup':>6}")
f1s_blind = []
for r in rels_blind:
    p_ = tp_b[r] / (tp_b[r] + fp_b[r]) if (tp_b[r] + fp_b[r]) else 0
    rc = tp_b[r] / (tp_b[r] + fn_b[r]) if (tp_b[r] + fn_b[r]) else 0
    f1 = 2 * p_ * rc / (p_ + rc) if (p_ + rc) else 0
    f1s_blind.append(f1)
    print(f"{r:<22}{p_:>8.3f}{rc:>8.3f}{f1:>8.3f}{gc_blind[r]:>6}")
macro_f1_blind = sum(f1s_blind) / len(f1s_blind)
print("-" * 52)
print(f"MACRO F1 (ciego): {macro_f1_blind:.4f}  |  gold={len(gold_rel)}  pred={len(pred_rel)}")

print(f"\n{'':<30}{'curado (dev)':>18}{'ciego (blind)':>18}")
print(f"{EXPERIMENT_NAME:<30}{macro_f1:>18.4f}{macro_f1_blind:>18.4f}")
print(f"{'Baseline mBERT (referencia)':<30}{'0.6944':>18}{'0.3126':>18}")
print(f"caida curado->ciego: {macro_f1_blind - macro_f1:+.4f}")

json.dump({
    "experiment": EXPERIMENT_NAME,
    "model": MODEL_NAME,
    "epochs": EPOCHS,
    "macro_f1_curado": macro_f1,
    "macro_f1_ciego": macro_f1_blind,
    "blind_candidates": len(blind_raw),
    "gold_relations": len(gold_rel),
    "inference_time_min": elapsed_blind / 60,
}, open(BLIND_RESULTS_PATH, "w"), indent=2)
print("guardado:", BLIND_RESULTS_PATH.name)

---
## 11. Calibración de umbral en blind (post-hoc, sin reentrenar)

**Motivación:** con argmax puro, `no_relation` compite en desventaja contra las otras 14 clases a la vez, lo que suele hacer que el modelo sobre-prediga relación en el protocolo blind. Aquí probamos si, sin reentrenar, basta con exigir más confianza que el argmax puro para predecir una relación en vez de `no_relation`.

Reutiliza de la sección 10 (validación blind): `infer_model`, `encoder_pred`, `blind_raw`, `rel2id`, `id2rel`, `gold_rel`, `rels_blind`, `_key`, `macro_f1_blind`, `BLIND_BATCH_SIZE`, `device`, `OUTPUT_DIR`, `EXPERIMENT_NAME`.

In [ ]:
# =====================================================================
#  1) INFERENCIA CON DISTRIBUCION COMPLETA (no solo el argmax)
#  Se guarda a disco para poder probar cualquier threshold despues SIN
#  volver a pasar los 282k candidatos por la GPU.
# =====================================================================
NO_REL_ID = rel2id["no_relation"]
PROBS_PATH = OUTPUT_DIR / f"blind_probs_{EXPERIMENT_NAME}.npy"

def batched_predict_probs(infer_model, encoder, instances, batch_size, device):
    """Igual que batched_predict, pero guarda las 15 probabilidades de cada
    candidato en vez de quedarnos solo con el argmax."""
    all_probs = np.zeros((len(instances), len(rel2id)), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(instances), batch_size):
            batch = instances[start:start + batch_size]
            tok = [encoder.tokenize({"text": i["text"], "h": {"pos": i["h"]["pos"]}, "t": {"pos": i["t"]["pos"]}}) for i in batch]
            fields = [torch.cat([t[k] for t in tok], dim=0).to(device) for k in range(len(tok[0]))]
            logits = infer_model(*fields)
            probs = torch.softmax(logits, dim=-1)
            all_probs[start:start + len(batch)] = probs.cpu().numpy()
            done = min(start + batch_size, len(instances))
            if done % (batch_size * 50) == 0 or done == len(instances):
                print(f"  {done}/{len(instances)}", flush=True)
    return all_probs

t0 = time.time()
blind_probs = batched_predict_probs(infer_model, encoder_pred, blind_raw, BLIND_BATCH_SIZE, device)
print(f"inferencia con distribucion completa: {(time.time() - t0) / 60:.1f} min")

np.save(PROBS_PATH, blind_probs)
print(f"guardado: {PROBS_PATH.name} (shape={blind_probs.shape})")

In [ ]:
# =====================================================================
#  2) BARRIDO DE THRESHOLD
#  Regla nueva: predice la mejor relacion distinta de no_relation SOLO si
#  su probabilidad supera `threshold`; si no, predice no_relation.
# =====================================================================
other_ids = np.array([i for i in range(len(rel2id)) if i != NO_REL_ID])

def preds_at_threshold(probs, threshold):
    other_probs = probs[:, other_ids]
    best_other_local = other_probs.argmax(axis=1)
    best_other_score = other_probs.max(axis=1)
    best_other_class = other_ids[best_other_local]
    return np.where(best_other_score > threshold, best_other_class, NO_REL_ID)

def macro_f1_for_preds(pred_ids):
    pred_labels = [id2rel[i] for i in pred_ids]
    rows = [
        {"document_id": inst["doc_id"], "relation": rel, "head_span": inst["head_span"], "tail_span": inst["tail_span"]}
        for inst, rel in zip(blind_raw, pred_labels) if rel != "no_relation"
    ]
    pred_map = {_key(r["document_id"], r["head_span"], r["tail_span"]): r["relation"] for r in rows}

    tp, fp, fn = Counter(), Counter(), Counter()
    for k, pr in pred_map.items():
        gr = gold_rel.get(k)
        if gr is None: fp[pr] += 1
        elif pr == gr: tp[pr] += 1
        else: fp[pr] += 1; fn[gr] += 1
    for k, gr in gold_rel.items():
        if k not in pred_map: fn[gr] += 1

    f1s, precs, recs = [], [], []
    for r in rels_blind:
        p_ = tp[r] / (tp[r] + fp[r]) if (tp[r] + fp[r]) else 0
        rc = tp[r] / (tp[r] + fn[r]) if (tp[r] + fn[r]) else 0
        f1 = 2 * p_ * rc / (p_ + rc) if (p_ + rc) else 0
        f1s.append(f1); precs.append(p_); recs.append(rc)
    return sum(f1s) / len(f1s), sum(precs) / len(precs), sum(recs) / len(recs), len(pred_map)

thresholds = np.arange(0.05, 0.96, 0.05)
sweep_rows = []
for th in thresholds:
    pred_ids = preds_at_threshold(blind_probs, th)
    f1_th, p_th, r_th, n_pred = macro_f1_for_preds(pred_ids)
    sweep_rows.append({"threshold": float(th), "macro_f1": f1_th, "macro_precision": p_th, "macro_recall": r_th, "n_predicted": n_pred})
    print(f"th={th:.2f}  MacroF1={f1_th:.4f}  MacroP={p_th:.4f}  MacroR={r_th:.4f}  n_pred={n_pred}")

sweep_df = pd.DataFrame(sweep_rows)
best_row = sweep_df.loc[sweep_df["macro_f1"].idxmax()]
print("\nMejor threshold:", best_row.to_dict())

In [ ]:
# =====================================================================
#  3) CURVA P/R/F1 EN FUNCION DEL THRESHOLD
# =====================================================================
FIG_DIR = OUTPUT_DIR / "figs"; FIG_DIR.mkdir(exist_ok=True, parents=True)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(sweep_df["threshold"], sweep_df["macro_f1"], marker="o", label="Macro F1")
ax.plot(sweep_df["threshold"], sweep_df["macro_precision"], marker="s", alpha=0.7, label="Macro Precision")
ax.plot(sweep_df["threshold"], sweep_df["macro_recall"], marker="^", alpha=0.7, label="Macro Recall")
ax.axvline(best_row["threshold"], color="red", linestyle="--", label=f"mejor threshold={best_row['threshold']:.2f}")
ax.axhline(macro_f1_blind, color="gray", linestyle=":", label=f"argmax puro (actual)={macro_f1_blind:.4f}")
ax.set_xlabel("threshold (prob. minima para predecir relacion en vez de no_relation)")
ax.set_ylabel("score")
ax.set_title(f"Calibracion de umbral en blind - {EXPERIMENT_NAME}")
ax.legend(); ax.grid(alpha=0.3)
fig.savefig(FIG_DIR / f"threshold_sweep_{EXPERIMENT_NAME}.png", dpi=150, bbox_inches="tight")
plt.show()

sweep_df.to_csv(OUTPUT_DIR / f"threshold_sweep_{EXPERIMENT_NAME}.csv", index=False)
print("guardado csv y figura del barrido de threshold")

In [ ]:
# =====================================================================
#  4) RESULTADO FINAL CON EL MEJOR THRESHOLD vs. ARGMAX PURO
# =====================================================================
best_threshold = float(best_row["threshold"])
final_pred_ids = preds_at_threshold(blind_probs, best_threshold)
final_f1, final_p, final_r, n_final_pred = macro_f1_for_preds(final_pred_ids)

print(f"{'':<35}{'argmax puro':>15}{'con threshold':>15}")
print(f"{'Macro F1 (blind)':<35}{macro_f1_blind:>15.4f}{final_f1:>15.4f}")
print(f"{'Macro Precision (blind)':<35}{'-':>15}{final_p:>15.4f}")
print(f"{'Macro Recall (blind)':<35}{'-':>15}{final_r:>15.4f}")
print(f"{'Predicciones de relacion':<35}{len(pred_rel):>15}{n_final_pred:>15}")

json.dump({
    "experiment": EXPERIMENT_NAME,
    "calibration": "global_threshold_no_relation_vs_rest",
    "best_threshold": best_threshold,
    "macro_f1_argmax": macro_f1_blind,
    "macro_f1_threshold": final_f1,
    "macro_precision_threshold": final_p,
    "macro_recall_threshold": final_r,
    "n_predicted_argmax": len(pred_rel),
    "n_predicted_threshold": n_final_pred,
    "sweep": sweep_rows,
}, open(OUTPUT_DIR / f"results_threshold_calibration_{EXPERIMENT_NAME}.json", "w"), indent=2)
print("\nguardado: results_threshold_calibration json")

---
## 12. Comparacion con Exp 1A (PubMedBERT) y verificacion con el scorer oficial

**Motivo:** el notebook 1A (`1A-experimentopubmedbert.ipynb`) entrena PubMedBERT con
**exactamente los mismos hiperparametros** que este (`max_length=256, batch=16, lr=2e-5,
epochs=15, warmup=300, neg_ratio=3, seed=42`) y el mismo protocolo de evaluacion
(curado + ciego + calibracion de threshold). Es la comparacion 1:1 real entre backbones
-- las cifras "PubMedBERT (Exp1)" de la cabecera de este notebook vienen de una corrida
distinta (10 epochs) y no son directamente comparables.

Reutiliza variables ya definidas en este notebook: `macro_f1` (seccion 6), `macro_f1_blind`
(seccion 10), `best_threshold`/`final_f1` (seccion 11), `OUTPUT_DIR`, `EXPERIMENT_NAME`,
`blind_raw`, `final_pred_ids`, `id2rel`, `BLIND_PRED_PATH`, `BLIND_GOLD_TSV`.

In [ ]:
# ============================================================
# TABLA COMPARATIVA 1A (PubMedBERT) vs 1B (BioLinkBERT-base)
# Carga los resultados ya guardados de 1A y los compara con los de
# este notebook (ya en memoria: macro_f1, macro_f1_blind, final_f1).
# ============================================================
EXP1A_DIR = Path("../outputs/1A-pubmedbert")

with open(EXP1A_DIR / "results_pubmedbert.json") as f:
    res_1a = json.load(f)
with open(EXP1A_DIR / "results_blind_pubmedbert.json") as f:
    blind_1a = json.load(f)
with open(EXP1A_DIR / "results_threshold_calibration_pubmedbert.json") as f:
    thresh_1a = json.load(f)

with open(OUTPUT_DIR / f"results_threshold_calibration_{EXPERIMENT_NAME}.json") as f:
    thresh_1b = json.load(f)

comparativa = [
    ("Macro F1 curado", res_1a["results"]["macro_f1"], macro_f1),
    ("Macro F1 ciego (argmax)", blind_1a["macro_f1_ciego"], macro_f1_blind),
    ("Macro F1 ciego (threshold=0.95)", thresh_1a["macro_f1_threshold"], thresh_1b["macro_f1_threshold"]),
]

print(f"{'Metrica':<32}{'1A PubMedBERT':>15}{'1B BioLinkBERT':>16}{'Diff':>10}")
print("-" * 73)
for name, a, b in comparativa:
    print(f"{name:<32}{a:>15.4f}{b:>16.4f}{b - a:>+10.4f}")

# Diferencia por relacion en curado (backbone identico, misma configuracion)
print(f"\n{'Relacion':<22}{'1A F1':>10}{'1B F1':>10}{'Diff':>10}")
print("-" * 52)
for rel in sorted(results.keys()):
    if rel == "no_relation":
        continue
    f1_1a = res_1a["results"]["per_relation"][rel]["f1"]
    f1_1b = results[rel]["f1"]
    print(f"{rel:<22}{f1_1a:>10.4f}{f1_1b:>10.4f}{f1_1b - f1_1a:>+10.4f}")

In [ ]:
# ============================================================
# VERIFICACION CON EL SCORER OFICIAL (baseline/score.py)
# Hasta ahora el resultado con threshold calibrado (seccion 11) solo se
# habia calculado a mano en Python -- nunca se habia exportado a TSV ni
# pasado por el script oficial. Aqui se guarda esa prediccion y se
# llama a baseline/score.py sobre ambos TSV (argmax puro y threshold)
# para confirmar que la reimplementacion manual coincide exactamente
# con el criterio oficial de CodaBench.
# ============================================================
import subprocess

BLIND_PRED_TH_PATH = OUTPUT_DIR / f"eng_pred_blind_th{best_threshold:.2f}_{EXPERIMENT_NAME}.tsv"

final_pred_labels = [id2rel[i] for i in final_pred_ids]
rows_th = [
    {
        "document_id": inst["doc_id"], "relation": rel,
        "head_text": inst["h"]["name"], "head_span": inst["head_span"], "head_type": inst["head_type"],
        "tail_text": inst["t"]["name"], "tail_span": inst["tail_span"], "tail_type": inst["tail_type"],
    }
    for inst, rel in zip(blind_raw, final_pred_labels) if rel != "no_relation"
]
pd.DataFrame(rows_th).to_csv(BLIND_PRED_TH_PATH, sep="\t", index=False)
print(f"guardado: {BLIND_PRED_TH_PATH.name} ({len(rows_th)} predicciones)")

for label, pred_path in [("argmax puro", BLIND_PRED_PATH), (f"threshold={best_threshold:.2f}", BLIND_PRED_TH_PATH)]:
    out = subprocess.run(
        ["python", "../baseline/score.py", "--pred", str(pred_path), "--gold", str(BLIND_GOLD_TSV)],
        capture_output=True, text=True,
    )
    macro_line = next(l for l in out.stdout.splitlines() if l.startswith("Macro F1"))
    print(f"{label:<20} -> {macro_line}  (score.py oficial)")

### Interpretacion

**BioLinkBERT-base gana a PubMedBERT en los tres escenarios**, de forma consistente
pero modesta (curado +0.006, ciego argmax +0.019, ciego con threshold +0.030). La
ventaja crece segun el protocolo se acerca al real, lo que sugiere que el
pre-entrenamiento con enlaces entre documentos aporta algo mas en el escenario
realista que en el curado -- aunque la magnitud es pequena para ser concluyente
con un unico seed.

**Dato relevante para la memoria:** en ciego sin calibrar, *ambos* backbones
biomedicos (PubMedBERT 0.2894, BioLinkBERT 0.3079) quedan por debajo del rango del
baseline mBERT citado en la seccion 10 (0.3126-0.3435). Solo tras calibrar el
threshold, BioLinkBERT (0.4286) supera claramente ese rango; PubMedBERT (0.3986) se
queda en el borde. Es decir, la superioridad de los backbones biomedicos frente al
baseline generico solo se sostiene si se corrige la sobre-prediccion de relacion en
el protocolo ciego -- con argmax puro no se sostiene.

Por relacion (curado), la ganancia neta de BioLinkBERT viene sobre todo de
`ALTERNATIVE_NAME` (0.346 -> 0.500) y mejoras menores en `PART_OF`, `SUBCLASS_OF`,
`HAS_CAUSE`, `APPLIED_TO`; se compensa con perdidas en `FINDING_OF` (0.743 -> 0.680),
`USED_IN`, `TREATED_USING` y `TO_DETECT_OR_STUDY`.

**Verificacion con el scorer oficial:** la celda anterior confirma que
`baseline/score.py` reproduce exactamente los mismos valores que las secciones 10 y
11 calculan a mano (mismo criterio de clave `document_id|head_span|tail_span`, misma
contabilidad de TP/FP/FN) -- la reimplementacion del notebook es fiel al criterio
oficial de CodaBench.

**Limitacion conocida del scorer (compartida por `score.py` y esta
reimplementacion):** 9 de los 2891 pares en `eng-dev-rel.tsv` tienen **dos relaciones
gold distintas** sobre el mismo `(document_id, head_span, tail_span)` -- p.ej. un par
anotado a la vez como `PART_OF` y `SUBCLASS_OF`. Como el gold se indexa con un
diccionario simple `{clave: relacion}`, la segunda anotacion sobrescribe a la
primera y solo una de las dos puede contar como acierto. El impacto es minimo
(~0.3% de las instancias), pero es un techo de recall estructural del formato --no
un bug del pipeline-- que conviene dejar anotado si se discute el detalle de la
metrica en la memoria.

**Nota de alcance:** esta comparacion y la verificacion con `score.py` solo cubren
el protocolo ciego (secciones 10-12). La evaluacion curada de las secciones 5-7 usa
un matching por posicion en listas en memoria, no el join por clave que hace
`score.py` -- es valida para iterar rapido, pero no es la superficie que puntuaria
CodaBench.